# Step 1: Setup up dataset AND constant values [DONE for wiki, need to do for ambig]
# Step 2: Convert Query to n-Query [Maybe, need more testing issues - how to convert llm reposon to n-query?]
# Step 3: Create a function to combine n-query doc ranking to k-doc ranking [DONE]
# Step 4: Setup the rag pipeline

In [1]:
import pyterrier as pt
from pyterrier.measures import *
from pyterrier_t5 import MonoT5ReRanker

from code_base.wiki_loader_pyt import wiki_loader_pyt

# Constants
DATA_PATH = "/mnt/c/Users/prath/Documents/Uni(Tudelft)/MSc/MasterThesis/data/2wikimultihopQA"
SPLIT = "dev"

# Load dataframes
docs_df, topics_df, qrels_df = wiki_loader_pyt(DATA_PATH, SPLIT)

/home/prat/miniconda3/envs/mt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing corpus...


100%|██████████| 563424/563424 [00:00<00:00, 2010776.86it/s]


Created docs_df with 495034 documents.

Processing dev set for topics and qrels...


100%|██████████| 12576/12576 [00:00<00:00, 228041.38it/s]


Created topics_df with 12576 queries.
Created qrels_df with 125760 relevance judgments.


In [2]:
docs_df = docs_df.dropna(subset=['docno', 'text'])
docs_df['docno'] = docs_df['docno'].astype(str)
docs_df = docs_df[docs_df['text'].str.strip() != '']
docs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495034 entries, 0 to 495033
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   docno   495034 non-null  object
 1   text    495034 non-null  object
dtypes: object(2)
memory usage: 7.6+ MB


In [3]:
#Uncomment if you want to reindex
indexer = pt.IterDictIndexer("./index/wiki_musique_index", overwrite=True, meta={'docno': 256, 'text': 4096})
index_ref = indexer.index(docs_df.to_dict(orient='records'))

# index_path = "./index/wiki_musique_index/data.properties"  # path to the Terrier index metadata
# index_ref = pt.IndexFactory.of(index_path)


print(index_ref)

Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


14:05:58.717 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (Timeline of the Insurgency in Northeast India) - further warnings are suppressed
14:09:45.676 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 2 empty documents
<org.terrier.querying.IndexRef at 0x73232f0223f0 jclass=org/terrier/querying/IndexRef jself=<LocalRef obj=0x1dc38380 at 0x7323848a7ad0>>


In [4]:
def preprocess_query(query):
    # remove punctuation and special characters
    query = ''.join(e for e in query if e.isalnum() or e.isspace())
    # convert to lowercase
    query = query.lower()
    return query

topics_df['query'] = topics_df['query'].apply(preprocess_query)

In [ ]:

c=10
bm25 = pt.terrier.Retriever(index_ref, wmodel="BM25", controls={})

pipline= bm25 % c  >> pt.text.get_text(index_ref, "text") >> MonoT5ReRanker(verbose=False, batch_size=16)

experiment = pt.Experiment(
    [pipline],
    topics_df,
    qrels_df,
    batch_size=10,
    eval_metrics=["map", "ndcg", R@5],
    names=["BM25+MonoT5"],
    verbose=True,
    save_dir=DATA_PATH,
)

# experiment = pt.Experiment(
#     [bm25 % c, pipline],
#     topics_df,
#     qrels_df,
#     batch_size=10,
#     eval_metrics=["map", "ndcg", R@5],
#     names=["BM25","BM25+MonoT5"],
#     verbose=True,
#     save_dir=DATA_PATH,
# )

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
pt.Experiment:   4%|▍         | 54/1258 [02:43<58:36,  2.92s/batches]  

In [19]:
experiment

,name,map,ndcg
0,BM25,0.216017,0.429157


retrival dataset - Bright dataset

